In [ ]:
import pandas as pd

# Carregar e pré-processar os dados
data = pd.read_csv('phishing_email.csv')

# Renomear as colunas e remover a coluna 'Unnamed: 0'
data = data.rename(columns={'Email Text': 'email_text', 'Email Type': 'email_type'}).drop(columns=['Unnamed: 0'])

# Remover linhas com valores vazios
data = data.dropna(subset=['email_text', 'email_type'])

# Salvar o novo dataset após o pré-processamento
data.to_csv('phishing_email.csv', index=False)

# Carregar e pré-processar os dados
data = pd.read_csv('phishing_email.csv')

# Corrigir o mapeamento de 'Safe Email' e 'Phishing Email' para 0 e 1
data['email_type'] = data['email_type'].map({'Safe Email': 0, 'Phishing Email': 1})

# Remover valores nulos na coluna 'email_type' (se necessário)
data = data.dropna(subset=['email_type'])

# Verificar se a coluna foi mapeada corretamente
print(data.head())

# Salvar o novo dataset como CSV
data.to_csv('phishing_email.csv', index=False)




In [ ]:
import pandas as pd

# Carregar o arquivo test_data.csv
test_data = pd.read_csv('test_data.csv')

# Dividir o arquivo: 500 dados para test_api.csv e o restante para test_data.csv
test_api = test_data.sample(n=500, random_state=42)  # Seleciona aleatoriamente 500 amostras
test_data = test_data.drop(test_api.index)           # Remove essas amostras do test_data

test_api = test_api.drop(columns=['email_type'])

# Salvar os novos arquivos
test_api.to_csv('test_api.csv', index=False)
test_data.to_csv('test_data.csv', index=False)




In [ ]:
import pickle
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.optimizers import Adam

# Carregar e pré-processar os dados
data = pd.read_csv('phishing_email.csv')

# Definir as variáveis de entrada e saída
X = data['email_text']  # Texto do email
y = data['email_type']  # Rótulos (0: seguro, 1: phishing)

# Dividir os dados em conjuntos de treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Criar DataFrames para os conjuntos de treino e teste
train_df = pd.DataFrame({'email_text': X_train, 'email_type': y_train})
test_df = pd.DataFrame({'email_text': X_test, 'email_type': y_test})

# Salvar os arquivos CSV para posterior uso
train_df.to_csv('train_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)

print("Conjuntos de treino e teste salvos com sucesso!")

# Transformar o texto (email_text) em vetores usando TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Reduzir a taxa de aprendizado
optimizer = Adam(learning_rate=0.0001)  # Ajuste da taxa de aprendizado

# Definir o modelo de rede neural
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(X_train_tfidf.shape[1],)),
    tf.keras.layers.Dense(512, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(256, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Saída binária com 'sigmoid'
])

# Compilar o modelo
model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Treinamento do modelo
num_epochs = 2000
history = model.fit(X_train_tfidf, y_train, epochs=num_epochs, validation_data=(X_test_tfidf, y_test), verbose=1)

# Converter histórico em DataFrame para análise posterior
history_df = pd.DataFrame(history.history)
history_df['epoch'] = history_df.index + 1

# Calcular as médias de acurácia e perda
accuracy_values = history.history['accuracy']
loss_values = history.history['loss']

avg_accuracy = np.mean(accuracy_values)
avg_loss = np.mean(loss_values)

# Exibir as médias
print(f"Média de Acurácia: {avg_accuracy:.4f}")
print(f"Média de Perda (Loss): {avg_loss:.4f}")

# Função para plotar gráficos de métricas
def plot_metrics(history_df, metric, title, ylabel):
    plt.figure(figsize=(10, 5))
    plt.plot(history_df['epoch'], history_df[metric], label=f'Treino {ylabel}')
    plt.plot(history_df['epoch'], history_df[f'val_{metric}'], label=f'Validação {ylabel}')
    plt.xlabel('Época')
    plt.ylabel(ylabel)
    plt.legend()
    plt.title(title)
    plt.show()

# Gráficos de métricas
plot_metrics(history_df, 'loss', 'Perda (Binary Crossentropy) por Época', 'Loss')
plot_metrics(history_df, 'accuracy', 'Acurácia por Época', 'Acurácia')

with open('phishing_model.pkl', 'wb') as model_file:
  pickle.dump(model, model_file)

with open('vectorizer.pkl', 'wb') as vectorizer_file:
  pickle.dump(vectorizer, vectorizer_file)


# Carregar o arquivo test_api.csv
test_api_data = pd.read_csv('test_api.csv')

# Verificar se a coluna 'email_text' está presente
if 'email_text' in test_api_data.columns:
    # Transformar o texto dos emails em vetores usando o vetorizer carregado
    X_test_api = vectorizer.transform(test_api_data['email_text'])
    
    # Fazer previsões usando o modelo carregado
    predictions = model.predict(X_test_api)
    
    # Adicionar as previsões ao DataFrame
    test_api_data['prediction'] = predictions
    test_api_data['prediction'] = test_api_data['prediction'].apply(lambda x: 'phishing' if x == 1 else 'safe')
    
    # Salvar as previsões em um novo arquivo CSV
    test_api_data[['email_text', 'prediction']].to_csv('predicoes.csv', index=False)
    print("As previsões foram salvas em 'predicoes.csv'")
else:
    print("A coluna 'email_text' não está presente no arquivo test_api.csv.")

